In [1]:
import pandas as pd
from pandas.errors import PerformanceWarning
import warnings

warnings.simplefilter('ignore', category=PerformanceWarning)

In [2]:
from src.features.load_intra_day_dataset import load_intra_day_dataset

start = pd.Timestamp("2024-07-01")
end = pd.Timestamp("2025-04-01")
val_cutoff = pd.Timestamp("2025-01-01")
select_cutoff = pd.Timestamp("2025-02-01")
test_cutoff = pd.Timestamp("2025-03-01")

df = load_intra_day_dataset(start, end)

2025-04-14 23:17:50,489 - INFO - Loading intraday data
2025-04-14 23:17:50,490 - INFO - Loading data from /Users/pavelholman/FIT/bp/data/interim/data.csv


In [3]:
from src.models.data_split import time_series_split, ID_KNOWN
from src.models.train_intra_day_time_series import train_intra_day_time_series

ts_train, ts_val, ts_select, ts_test = time_series_split(df, val_cutoff, select_cutoff, test_cutoff)
ts_metrics = ["MAE", "RMSE"]

ts_mae_model = train_intra_day_time_series(ts_train, ts_val, "MAE", cache="MAE")
display(ts_mae_model.leaderboard(ts_select, extra_metrics=ts_metrics))

ts_rmse_model = train_intra_day_time_series(ts_train, ts_val, "RMSE", cache="RMSE")
display(ts_rmse_model.leaderboard(ts_select, extra_metrics=ts_metrics))

/Users/pavelholman/FIT/bp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-14 23:17:53,145 - INFO - Data split info:
Original DataFrame size: 25632
Train size: 16992 (66.29%)
Validation size: 2976 (11.61%)
Select size: 2688 (10.49%)
Test size: 2976 (11.61%)
2025-04-14 23:17:53,146 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/intra_day/time/MAE
Loading predictor from path /Users/pavelholman/FIT/bp/models/intra_day/time/MAE
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-7.160570,-12.703469,0.959461,1.147792,2.001209,10,-7.160570,-8.422258
1,AutoARIMA,-10.236626,-30.799013,196.385824,220.410509,0.076159,9,-10.236626,-12.451476
2,WeightedEnsemble,-10.542175,-8.239339,6.522441,14.713415,0.320415,11,-10.542175,-12.110641
3,SeasonalNaive,-10.866250,-17.937396,0.098096,0.779555,0.008281,2,-10.866250,-13.967777
4,SeasonalAverage,-11.535000,-16.853125,1.162591,0.148992,0.006957,4,-11.535000,-13.604641
5,Naive,-13.930104,-31.582083,0.088883,1.422431,0.008854,1,-13.930104,-18.094960
6,ETS,-13.962541,-19.620110,0.970652,8.380315,0.102330,7,-13.962541,-18.211973
7,AutoETS,-13.962541,-19.620110,1.016174,7.694178,0.038894,8,-13.962541,-18.211973
8,Average,-14.755417,-24.066458,0.795768,0.062268,0.006770,3,-14.755417,-16.413556
9,RecursiveTabular,-25.236122,-22.128992,3.222123,2.951796,59.742136,5,-25.236122,-29.457607


2025-04-14 23:17:55,180 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/intra_day/time/RMSE
Loading predictor from path /Users/pavelholman/FIT/bp/models/intra_day/time/RMSE
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-8.422258,-17.211976,1.165157,1.085254,0.006164,10,-7.160570,-8.422258
1,WeightedEnsemble,-9.349842,-14.409925,6.216618,11.526081,0.331451,11,-7.485606,-9.349842
2,AutoARIMA,-12.451476,-36.853119,181.194125,214.105201,0.047046,9,-10.236626,-12.451476
3,SeasonalAverage,-13.685394,-18.258891,0.158136,0.193195,0.007817,4,-11.253723,-13.685394
4,SeasonalNaive,-13.967777,-24.948692,1.070772,0.086814,0.006652,2,-10.866250,-13.967777
5,DirectTabular,-16.314194,-25.376380,1.482311,1.279909,30.325427,6,-14.670991,-16.314194
6,Average,-17.699087,-26.698662,0.051753,0.058028,0.006811,3,-15.790191,-17.699087
7,Naive,-18.094960,-39.960046,0.088298,0.224390,0.008415,1,-13.930104,-18.094960
8,ETS,-18.211973,-25.953065,0.800523,6.918969,0.058551,7,-13.962541,-18.211973
9,AutoETS,-18.211973,-25.953065,0.959768,6.933908,0.169201,8,-13.962541,-18.211973


In [4]:
from src.models.data_split import tabular_split
from src.models.train_intra_day_tabular import train_intra_day_tabular

tab_df = df[[col for col in df.columns if any(col.startswith(prefix) for prefix in ID_KNOWN)] + ["ID Weighted average of the prices (EUR/MWh)"]]
tab_train, tab_val, tab_select, tab_test = tabular_split(tab_df, val_cutoff, select_cutoff, test_cutoff)
tab_metrics = ["mae", "rmse"]

tab_mae_model = train_intra_day_tabular(tab_train, tab_val, "mae", cache="MAE")
display(tab_mae_model.leaderboard(tab_test, extra_metrics=tab_metrics))

tab_rmse_model = train_intra_day_tabular(tab_train, tab_val, "rmse", cache="RMSE")
display(tab_rmse_model.leaderboard(tab_test, extra_metrics=tab_metrics))

/Users/pavelholman/FIT/bp/src/models/data_split.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["item_id"] = 1
2025-04-14 23:17:57,398 - INFO - Data split info:
Original DataFrame size: 25632
Train size: 16992 (66.29%)
Validation size: 2976 (11.61%)
Select size: 2688 (10.49%)
Test size: 2976 (11.61%)
2025-04-14 23:17:57,399 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/intra_day/tabular/MAE


,model,score_test,mean_absolute_error,root_mean_squared_error,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,-14.570870,-14.570870,-21.918670,-28.067439,mean_absolute_error,0.621897,0.168787,999.643727,0.621897,0.168787,999.643727,1,True,2
1,WeightedEnsemble_L2,-14.962723,-14.962723,-21.260110,-25.749719,mean_absolute_error,1.714290,1.193962,1026.423993,0.002212,0.000379,0.013874,2,True,5
2,XGBoost,-15.124838,-15.124838,-23.464951,-28.651808,mean_absolute_error,0.134870,0.132141,25.909077,0.134870,0.132141,25.909077,1,True,4
3,CatBoost,-15.189371,-15.189371,-21.565220,-25.897368,mean_absolute_error,0.109344,0.048479,26.360034,0.109344,0.048479,26.360034,1,True,3
4,KNeighbors,-29.971654,-29.971654,-46.608548,-42.546982,mean_absolute_error,0.980837,0.976317,0.406358,0.980837,0.976317,0.406358,1,True,1


2025-04-14 23:17:59,287 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/intra_day/tabular/RMSE


,model,score_test,mean_absolute_error,root_mean_squared_error,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-21.417857,-14.978440,-21.417857,-48.406792,root_mean_squared_error,1.447908,1.093826,50.030880,0.002286,0.000239,0.005638,2,True,5
1,CatBoost,-21.553102,-15.170034,-21.553102,-48.605415,root_mean_squared_error,0.117787,0.045276,27.338915,0.117787,0.045276,27.338915,1,True,3
2,RandomForest,-21.918670,-14.570870,-21.918670,-51.677846,root_mean_squared_error,0.480598,0.284384,1001.485661,0.480598,0.284384,1001.485661,1,True,2
3,XGBoost,-31.839088,-22.597198,-31.839088,-55.465625,root_mean_squared_error,0.131275,0.130987,22.295323,0.131275,0.130987,22.295323,1,True,4
4,KNeighbors,-46.608548,-29.971654,-46.608548,-69.578005,root_mean_squared_error,1.196560,0.917323,0.391004,1.196560,0.917323,0.391004,1,True,1


In [5]:
from autogluon.timeseries import TimeSeriesDataFrame

ts_retrain = TimeSeriesDataFrame.from_data_frame(pd.concat([ts_train, ts_val, ts_select]))
ts_model = train_intra_day_time_series(ts_retrain, None, "MAE", cache="retrain", models={"Chronos": {}})
display(ts_model.leaderboard(ts_test, extra_metrics=ts_metrics))

2025-04-14 23:18:01,613 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/intra_day/time/retrain
Loading predictor from path /Users/pavelholman/FIT/bp/models/intra_day/time/retrain
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-45.247794,-7.16057,0.882806,1.23965,0.521426,1,-45.247794,-56.546641
